# 1.2 Resampling

Small demos of every resampling scheme from the slides: a single train/test
split, k-fold and stratified k-fold cross-validation, and the bootstrap
(including out-of-bag). Same customer table throughout, predicting
`churned`.

## Setup

Throughout this notebook (and every other notebook in this folder) we reuse
the **same 300-row synthetic customer table** (`data/customers.csv`, see
`data/generate_customers.py`): a small telecom/subscription dataset with a
continuous target (`monthly_spend`) for regression notebooks and a binary
target (`churned`) for classification notebooks, plus a block of numeric
features used for resampling, evaluation-metric, dimensionality-reduction
and clustering demos. The point isn't the dataset -- it's small and
synthetic on purpose -- the point is seeing each technique applied to
something you already recognise from the last notebook.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("common"))
sys.path.insert(0, os.path.abspath("../common"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

CUSTOMERS_PATH = "data/customers.csv" if os.path.exists("data/customers.csv") else "../data/customers.csv"
df = pd.read_csv(CUSTOMERS_PATH)
df.head()

,customer_id,age,region,tenure_months,income_k,monthly_usage,support_calls,discount_pct,satisfaction,monthly_spend,churned
0,C0001,45.7,South,90.7,78.1,8.3,2,0,8.12,39.35,0
1,C0002,29.5,North,54.9,58.4,27.3,6,5,5.80,51.69,0
2,C0003,51.0,West,28.6,73.3,13.9,5,0,5.48,37.75,1
3,C0004,53.3,West,22.9,106.5,13.3,6,10,9.25,34.31,1
4,C0005,18.6,West,45.7,35.5,67.2,6,15,7.26,52.28,0


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X = df[NUMERIC_FEATURES] if False else df[["age","tenure_months","income_k","monthly_usage","support_calls","discount_pct","satisfaction"]]
y = df["churned"]

print("Full data:", X.shape, " churn rate:", round(y.mean(), 3))

Full data: (300, 7)  churn rate: 0.34


## A single train/test split

The most basic resampling scheme: hold out a slice of the data the model
never sees during fitting, then score on that slice only.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
print("train:", X_train.shape, "  test:", X_test.shape)
print("train churn rate:", round(y_train.mean(), 3), " test churn rate:", round(y_test.mean(), 3))

model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
print("Test accuracy:", round(accuracy_score(y_test, model.predict(X_test)), 3))

train: (210, 7)   test: (90, 7)
train churn rate: 0.338  test churn rate: 0.344
Test accuracy: 0.667


Notice `stratify=y`: without it, a small test set can end up with a churn
rate quite different from the training set just by chance -- exactly the
"why stratify" point from the slides.

## k-fold and stratified k-fold cross-validation

One split gives one noisy estimate of performance. k-fold CV rotates the
held-out slice k times and averages, which is what the slides' "why?" and
"cautionary example" slides are about -- try `k=2` below and watch the
variance across folds.

In [4]:
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score

for k in (2, 5, 10):
    scores_plain = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=KFold(n_splits=k, shuffle=True, random_state=0))
    scores_strat = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=StratifiedKFold(n_splits=k, shuffle=True, random_state=0))
    print(f"k={k:2d}  plain KFold: {scores_plain.round(3)}  (std={scores_plain.std():.3f})")
    print(f"       stratified : {scores_strat.round(3)}  (std={scores_strat.std():.3f})")

k= 2  plain KFold: [0.713 0.693]  (std=0.010)
       stratified : [0.693 0.673]  (std=0.010)
k= 5  plain KFold: [0.667 0.75  0.683 0.733 0.667]  (std=0.035)
       stratified : [0.683 0.7   0.667 0.7   0.667]  (std=0.015)


k=10  plain KFold: [0.6   0.7   0.833 0.667 0.733 0.633 0.7   0.767 0.567 0.733]  (std=0.076)
       stratified : [0.7   0.667 0.667 0.733 0.667 0.7   0.7   0.8   0.6   0.733]  (std=0.050)


With `k=2`, each fold is half the data -- notice how much more the plain
(non-stratified) fold accuracies can jump around than the stratified ones,
the same instability the "2-fold cross-validation: a cautionary example"
slide warns about.

## The bootstrap (incl. out-of-bag)

Instead of splitting once, resample the training set **with replacement**
many times. Each bootstrap sample leaves out, on average, about 36.8% of
the original rows ("out-of-bag") -- exactly the number to check below.

In [5]:
rng = np.random.default_rng(0)
n = len(X)
n_boot = 200
oob_fractions = []
boot_accuracies = []

X_arr = X.reset_index(drop=True)
y_arr = y.reset_index(drop=True)

for b in range(n_boot):
    boot_idx = rng.integers(0, n, size=n)
    oob_idx = np.setdiff1d(np.arange(n), boot_idx)
    oob_fractions.append(len(oob_idx) / n)
    if len(oob_idx) == 0:
        continue
    m = LogisticRegression(max_iter=1000).fit(X_arr.iloc[boot_idx], y_arr.iloc[boot_idx])
    boot_accuracies.append(accuracy_score(y_arr.iloc[oob_idx], m.predict(X_arr.iloc[oob_idx])))

print("Average out-of-bag fraction:", round(np.mean(oob_fractions), 3), " (theory: 1/e =", round(np.exp(-1), 3), ")")
print("Mean OOB accuracy over", n_boot, "bootstraps:", round(np.mean(boot_accuracies), 3))

Average out-of-bag fraction: 0.367  (theory: 1/e = 0.368 )
Mean OOB accuracy over 200 bootstraps: 0.687


**Takeaway**: a single split is cheap but noisy; k-fold trades compute for a
less noisy estimate (and stratification keeps small/imbalanced folds
honest); the bootstrap gives you both a performance estimate *and* a
natural held-out set (OOB) without sacrificing any training rows on
purpose -- the same idea random forests reuse later in Section 3.